# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [33]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [34]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

openai = OpenAI()
openai.api_key = api_key

In [35]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [36]:
# Get gpt-4o-mini to answer, with streaming
messages = [{"role":"user", "content":question}]
stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages, stream=True)
response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    update_display(Markdown(response), display_id=display_handle.display_id)
    
# display(Markdown(response.choices[0].message.content))

The code you've provided uses a generator expression combined with a set comprehension to yield values related to the authors of a collection of books stored in the variable `books`. Let's break it down step-by-step:

1. **Context**: 
   - `books` is expected to be an iterable (likely a list) of dictionaries, where each dictionary represents a book and contains various attributes, one of which is "author."

2. **Set Comprehension**: 
   - `{book.get("author") for book in books if book.get("author")}` is a set comprehension.
   - This expression iterates over each `book` in `books` and extracts the value associated with the key "author" using `book.get("author")`.
   - The `if book.get("author")` part ensures that only books with a non-null (truthy) author value are included in the set. If a book does not have an author, `book.get("author")` returns `None`, which is considered falsy in Python, and that book will be excluded.

3. **Unique Authors**: 
   - The use of a set comprehension means that only unique authors will be collected. If multiple books by the same author are present in the `books` list, only one instance of that author will be stored in the resulting set.

4. **Yield from**: 
   - `yield from` is used in Python to yield all values from an iterable. In this case, it yields each unique author retrieved from the set comprehension.
   - This allows the code to be used within a generator function, where it can yield a series of unique author names one at a time, rather than returning all of them at once.

5. **Summary**: 
   - Overall, this code creates an iterator that yields the unique authors of the books, ignoring any books that do not have an author. Each author is yielded one by one, which might be useful if you're processing a large collection of books and only need to handle the authors incrementally.

### Example

If `books` looks like this:

python
books = [
    {"title": "Book A", "author": "Author 1"},
    {"title": "Book B", "author": "Author 2"},
    {"title": "Book C"},  # No author
    {"title": "Book D", "author": "Author 1"},
]


The code would yield:

- "Author 1"
- "Author 2"

And "Author 1" would only be yielded once, despite appearing for multiple books.

In [37]:
# Get Llama 3.2 to answer

ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

# Get gpt-4o-mini to answer, with streaming
messages = [{"role":"user", "content":question}]
response = ollama_via_openai.chat.completions.create(model="llama3.2", messages=messages)

display(Markdown(response.choices[0].message.content))

This line of code uses a combination of generator expressions and the `yield from` syntax to extract the authors' names from a list or dictionary of books.

Here's a breakdown:

- `{...for book in books if book.get("author")}` is a generator expression. It generates an iterator that yields books for which the "author" key exists.

- `book.get("author")` inside the generator expression would normally retrieve the value associated with the "author" key from each book, but since we're using a dictionary (because of `get()`) in Python 3.x this will raise a KeyError if the key does not exist. That's why it is likely that you have a loop that also checks for the existence of that key to ensure that nothing fails while iterating over your list.

- `yield from {...}`: This syntax takes an iterable (like another generator expression here) and yields its values one at a time, allowing it to be used as a part of other iterables. So in this case, we're yielding the results of our previous line but instead of returning them, we're passing them on to some loop that may want to use the `yield from` to iterate over another iterable.

In essence, without using `yield from`, you'd likely have a loop like this:

```python
for book in books:
    if book.get("author"):
        yield book["author"]
```

But with `yield from`, your code can be much more concise and is often preferred for clarity.